In [2]:
!pip install -q transformers datasets peft accelerate bitsandbytes groq beautifulsoup4

In [3]:
import os

# Your Groq API key
os.environ["GROQ_API_KEY"] = "enter_api_key"

# Your Hugging Face token
os.environ["HF_TOKEN"] = "enter_hf_token"

# Your Hugging Face username
HF_USERNAME = "enter_hf_username"

print("Done!")

Done!


In [6]:
from bs4 import BeautifulSoup

# Your HTML files are in the Kaggle input directory
# The path follows the format /kaggle/input/your-dataset-name/filename
html_files = [
    "/kaggle/input/datasets/EStG.html",
    "/kaggle/input/datasets/KStG.html",
    "/kaggle/input/datasets/UStG.html"
]

# Extract plain text from each HTML file
documents = {}
for filepath in html_files:
    filename = filepath.split("/")[-1]  # Get just the filename
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    soup = BeautifulSoup(content, "html.parser")
    text = soup.get_text(separator="\n")
    # Clean up excessive blank lines
    text = "\n".join(line for line in text.splitlines() if line.strip())
    documents[filename] = text
    print(f"Extracted {len(text)} characters from {filename}")

print(f"\nLoaded {len(documents)} documents total")

Extracted 1634980 characters from EStG.html
Extracted 369672 characters from KStG.html
Extracted 513676 characters from UStG.html

Loaded 3 documents total


In [8]:
import random
import json
import time
from groq import Groq

def split_into_chunks(text, chunk_size=1000, overlap=100):
    """
    Splits a long text into smaller overlapping chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Split all documents into chunks
all_chunks = []
for filename, text in documents.items():
    chunks = split_into_chunks(text)
    all_chunks.extend(chunks)
    print(f"{filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

# Sample 200 random chunks
random.seed(42)
sampled_chunks = random.sample(all_chunks, 200)

# Initialize Groq client
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def generate_qa(chunk):
    """
    Sends a chunk of legal text to Groq and asks it to generate
    a question and answer pair based on that text.
    """
    prompt = f"""Given the following excerpt from Austrian tax law, generate one clear question and a concise answer based strictly on the text.

Respond in JSON format like this:
{{"question": "...", "answer": "..."}}

Only return the JSON, nothing else.

Text:
{chunk}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=512
    )

    text = response.choices[0].message.content.strip()
    parsed = json.loads(text)
    return parsed

# Generate Q&A pairs
qa_pairs = []
failed = 0

print(f"\nGenerating Q&A pairs for {len(sampled_chunks)} chunks...\n")

for i, chunk in enumerate(sampled_chunks):
    print(f"[{i+1}/{len(sampled_chunks)}] Generating Q&A...")
    try:
        qa = generate_qa(chunk)
        qa_pairs.append(qa)
    except Exception as e:
        print(f"  -> ERROR: {e}")
        failed += 1
    time.sleep(2)

print(f"\nDone! Generated {len(qa_pairs)} Q&A pairs ({failed} failed)")
print("\nExample:")
print(qa_pairs[0])

EStG.html: 1817 chunks
KStG.html: 411 chunks
UStG.html: 571 chunks

Total chunks: 2799

Generating Q&A pairs for 200 chunks...

[1/200] Generating Q&A...
[2/200] Generating Q&A...
[3/200] Generating Q&A...
[4/200] Generating Q&A...
[5/200] Generating Q&A...
[6/200] Generating Q&A...
[7/200] Generating Q&A...
[8/200] Generating Q&A...
[9/200] Generating Q&A...
[10/200] Generating Q&A...
[11/200] Generating Q&A...
[12/200] Generating Q&A...
[13/200] Generating Q&A...
[14/200] Generating Q&A...
[15/200] Generating Q&A...
[16/200] Generating Q&A...
[17/200] Generating Q&A...
[18/200] Generating Q&A...
[19/200] Generating Q&A...
[20/200] Generating Q&A...
[21/200] Generating Q&A...
[22/200] Generating Q&A...
[23/200] Generating Q&A...
[24/200] Generating Q&A...
[25/200] Generating Q&A...
[26/200] Generating Q&A...
[27/200] Generating Q&A...
[28/200] Generating Q&A...
[29/200] Generating Q&A...
[30/200] Generating Q&A...
[31/200] Generating Q&A...
[32/200] Generating Q&A...
  -> ERROR: Expec

In [9]:
import json

# Save the Q&A pairs to a file in the Kaggle working directory
with open("/kaggle/working/qa_pairs.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, ensure_ascii=False, indent=2)

print(f"Saved {len(qa_pairs)} Q&A pairs to qa_pairs.json")

Saved 198 Q&A pairs to qa_pairs.json


In [10]:
from datasets import Dataset

# Format each Q&A pair into a prompt/response format
# that the model will learn from during fine-tuning
def format_sample(qa):
    return {
        "text": f"### Question:\n{qa['question']}\n\n### Answer:\n{qa['answer']}"
    }

# Apply formatting to all Q&A pairs
formatted = [format_sample(qa) for qa in qa_pairs]

# Convert to a Hugging Face Dataset object (required for fine-tuning)
dataset = Dataset.from_list(formatted)

print(f"Dataset ready: {len(dataset)} samples")
print("\nExample formatted sample:")
print(dataset[0]["text"])

Dataset ready: 198 samples

Example formatted sample:
### Question:
Wenn der Leistungsempfänger eine juristische Person des öffentlichen Rechts oder ein Unternehmer ist, muss er die auf die Leistung entfallende Umsatzsteuer einbehalten und wohin abführen?

### Answer:
Ja, für das für den leistenden Unternehmer zuständige Finanzamt.


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Gemma 2B is much better than TinyLlama for German legal text
# It's still small enough to fit on Kaggle's free GPU
MODEL_NAME = "google/gemma-2b-it"

# Log in to Hugging Face (needed to download Gemma)
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # 16-bit precision to save memory
    device_map="auto"           # Automatically use the GPU
)

print(f"Model loaded!")
print(f"Number of parameters: {model.num_parameters():,}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading tokenizer...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded!
Number of parameters: 2,506,172,416


In [13]:
from peft import LoraConfig, get_peft_model, TaskType

# LoRA lets us fine-tune only a small part of the model
# instead of all parameters, saving memory and time
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                            # Rank — higher means more parameters to train
    lora_alpha=32,                  # Scaling factor for LoRA weights
    lora_dropout=0.1,               # Dropout to prevent overfitting
    target_modules=["q_proj", "v_proj"]  # Which layers to apply LoRA to
)

# Wrap the model with LoRA
model = get_peft_model(model, lora_config)

# Print how many parameters we are actually training
model.print_trainable_parameters()

trainable params: 921,600 || all params: 2,507,094,016 || trainable%: 0.0368


In [14]:
# Tokenization converts text into numbers the model can understand
def tokenize(sample):
    return tokenizer(
        sample["text"],
        truncation=True,      # Cut off if text is too long
        max_length=512,       # Maximum length in tokens
        padding="max_length"  # Pad shorter samples to the same length
    )

# Apply tokenization to the entire dataset
tokenized_dataset = dataset.map(tokenize, remove_columns=["text"])

# Set the format to PyTorch tensors (required for training)
tokenized_dataset.set_format("torch")

print(f"Tokenized dataset ready: {len(tokenized_dataset)} samples")

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

Tokenized dataset ready: 198 samples


In [15]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Training arguments control how the fine-tuning process works
training_args = TrainingArguments(
    output_dir="/kaggle/working/finetuned-tax-model",
    num_train_epochs=3,              # How many times to go through the dataset
    per_device_train_batch_size=4,   # How many samples to process at once
    gradient_accumulation_steps=4,   # Accumulate gradients to simulate larger batch
    warmup_steps=10,                 # Gradually increase learning rate at the start
    learning_rate=2e-4,              # How fast the model learns
    fp16=True,                       # Use 16-bit precision to save memory
    logging_steps=10,                # Print progress every 10 steps
    save_strategy="epoch",           # Save the model after each epoch
    report_to="none"                 # Don't report to any external service
)

# Data collator handles padding and batching during training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # False because we are doing causal LM not masked LM
)

# Set up the trainer with everything we prepared
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Start fine-tuning!
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning done!")

Starting fine-tuning...


Step,Training Loss
10,5.958477
20,4.791386
30,3.991519


Fine-tuning done!


In [16]:
from huggingface_hub import login

# Log in to Hugging Face
login(token=os.environ["HF_TOKEN"])

# Save the fine-tuned model to Hugging Face
model_name = f"{HF_USERNAME}/finetuned-austrian-tax-gemma"

print("Saving model to Hugging Face...")
model.push_to_hub(model_name)
tokenizer.push_to_hub(model_name)

print(f"Done! Model saved to: https://huggingface.co/{model_name}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Saving model to Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done! Model saved to: https://huggingface.co/huggingaccount123/finetuned-austrian-tax-gemma


In [20]:
import pandas as pd
import time

# Load the test dataset
df = pd.read_csv("/kaggle/input/datasets/dataset_clean.csv")
print(f"Dataset loaded: {len(df)} rows")

def ask_finetuned_model(question):
    """
    Formats the question the same way as our training data
    and generates an answer using the fine-tuned model.
    """
    prompt = f"### Question:\n{question}\n\n### Answer:\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode and extract only the answer part
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_output.split("### Answer:")[-1].strip()
    return answer

# Loop through all questions in batches
BATCH_SIZE = 8
prompts = [f"### Question:\n{row['prompt']}\n\n### Answer:\n" for _, row in df.iterrows()]
ids = [row["id"] for _, row in df.iterrows()]

results = []
print(f"\nStarting batched inference on {len(prompts)} questions...\n")

for i in range(0, len(prompts), BATCH_SIZE):
    batch_prompts = prompts[i:i+BATCH_SIZE]
    batch_ids = ids[i:i+BATCH_SIZE]

    print(f"[{i+BATCH_SIZE}/{len(prompts)}] Processing batch...")

    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    for j, output in enumerate(outputs):
        full_output = tokenizer.decode(output, skip_special_tokens=True)
        answer = full_output.split("### Answer:")[-1].strip()
        results.append({
            "id": batch_ids[j],
            "answer": answer
        })

# Save output CSV
output_df = pd.DataFrame(results)
output_df.to_csv("/kaggle/working/submission_step2_gemma.csv", index=False)

print(f"\nDone! Saved {len(output_df)} answers to 'submission_step2_gemma.csv'")
print("\nOutput preview:")
print(output_df.head(3))

Dataset loaded: 643 rows

Starting batched inference on 643 questions...

[8/643] Processing batch...
[16/643] Processing batch...
[24/643] Processing batch...
[32/643] Processing batch...
[40/643] Processing batch...
[48/643] Processing batch...
[56/643] Processing batch...
[64/643] Processing batch...
[72/643] Processing batch...
[80/643] Processing batch...
[88/643] Processing batch...
[96/643] Processing batch...
[104/643] Processing batch...
[112/643] Processing batch...
[120/643] Processing batch...
[128/643] Processing batch...
[136/643] Processing batch...
[144/643] Processing batch...
[152/643] Processing batch...
[160/643] Processing batch...
[168/643] Processing batch...
[176/643] Processing batch...
[184/643] Processing batch...
[192/643] Processing batch...
[200/643] Processing batch...
[208/643] Processing batch...
[216/643] Processing batch...
[224/643] Processing batch...
[232/643] Processing batch...
[240/643] Processing batch...
[248/643] Processing batch...
[256/643]